In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim
from libpysal.weights import KNN
from esda.moran import Moran
import numpy as np

def moran_score(pred, true):
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    # 避免除零：当 true == 0 时，如果 pred == 0 返回1，否则返回0
    # 但按照原公式，如果 true == 0 且 pred <= true (即 pred <= 0)，只有 pred=0 时成立，0/0 是 nan。
    # 我们可以计算两个分支，然后选择。
    with np.errstate(divide='ignore', invalid='ignore'):
        branch1 = pred / true
        branch2 = (1 - pred) / (1 - true)
    result = np.where(pred <= true, branch1, branch2)
    # 处理 true == 0 的情况
    if np.any(true == 0):
        # 对于 true == 0 且 pred == 0，结果应为 1
        # 对于 true == 0 且 pred > 0，原公式会走 branch2，得到 (1-pred)/(1-0) = 1-pred，这不太合理。
        # 但通常 true 不会为0，我们可以额外处理使结果在 [0,1] 且符合直觉。
        # 为了简单，我们仅处理 pred=0, true=0 -> 1，其他 true=0 -> 0
        zero_true_mask = (true == 0)
        result = np.where(zero_true_mask & (pred == 0), 1.0, result)
        result = np.where(zero_true_mask & (pred != 0), 0.0, result)
    return result

def evaluate_deconvolution(pred, true, coords=None, eps=1e-12,
                           ssim_grid_size=64, knn_k=6):
    """
    Evaluate deconvolution prediction against ground truth.

    Parameters:
        pred : np.ndarray, shape (n_spots, n_types)
        true : np.ndarray, shape (n_spots, n_types)
        coords : np.ndarray, shape (n_spots, 2), optional
            Spatial coordinates of spots. Required for Moran's I and SSIM.
        eps : float
        ssim_grid_size : int
            Grid size for rasterizing spatial maps (e.g., 64 -> 64x64).
        knn_k : int
            Number of neighbors for Moran's I spatial weight matrix.

    Returns:
        pd.DataFrame with all metrics.
    """
    pred = np.asarray(pred)
    true = np.asarray(true)
    assert pred.shape == true.shape, "Shape mismatch"
    n_spots, n_types = pred.shape

    # ---- 1. Per-spot PCC / Spearman ----
    spot_pcc, spot_spearman = [], []
    for i in range(n_spots):
        p, t = pred[i, :], true[i, :]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            spot_pcc.append(pcc)
            spot_spearman.append(spearman)
    mean_spot_pcc = np.nanmean(spot_pcc) if spot_pcc else np.nan
    mean_spot_spearman = np.nanmean(spot_spearman) if spot_spearman else np.nan

    # ---- 2. Per-cell-type PCC / Spearman ----
    type_pcc, type_spearman = [], []
    for j in range(n_types):
        p, t = pred[:, j], true[:, j]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            type_pcc.append(pcc)
            type_spearman.append(spearman)
    mean_type_pcc = np.nanmean(type_pcc) if type_pcc else np.nan
    mean_type_spearman = np.nanmean(type_spearman) if type_spearman else np.nan

    # ---- 3. MSE / RMSE ----
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)

    # ---- 4. Jensen-Shannon Divergence (per spot) ----
    jsd_list = []
    for i in range(n_spots):
        p = np.maximum(pred[i, :], 0)
        t = np.maximum(true[i, :], 0)
        if np.sum(p) < eps or np.sum(t) < eps:
            continue
        p = p / (np.sum(p) + eps)
        t = t / (np.sum(t) + eps)
        jsd = jensenshannon(p, t, base=2) ** 2
        jsd_list.append(jsd)
    mean_jsd = np.mean(jsd_list) if jsd_list else np.nan

    # ---- 5. Spatial metrics (SSIM + Moran's I) ----
    mean_ssim, mean_moran_pred, mean_moran_true, mean_moran_diff = np.nan, np.nan, np.nan, np.nan
    if coords is not None:
        coords = np.asarray(coords)

        # 5a. SSIM per cell type (rasterize spatial map to a 2D grid)
        ssim_list = []
        for j in range(n_types):
            pred_grid = _rasterize(coords, pred[:, j], grid_size=ssim_grid_size)
            true_grid = _rasterize(coords, true[:, j], grid_size=ssim_grid_size)
            # Skip if either grid is constant (SSIM undefined)
            if np.std(pred_grid) < eps or np.std(true_grid) < eps:
                continue
            # Use a fixed data range for both
            data_range = max(pred_grid.max(), true_grid.max()) - min(pred_grid.min(), true_grid.min())
            if data_range < eps:
                continue
            s = ssim(pred_grid, true_grid, data_range=data_range)
            ssim_list.append(s)
        mean_ssim = np.nanmean(ssim_list) if ssim_list else np.nan

        # 5b. Moran's I per cell type
        try:
            w = KNN.from_array(coords, k=knn_k)
            w.transform = 'r'  # row-standardized
            moran_pred_list, moran_true_list = [], []
            for j in range(n_types):
                p = pred[:, j]
                t = true[:, j]
                if np.std(p) > eps:
                    mi_p = Moran(p, w).I
                    moran_pred_list.append(mi_p)
                if np.std(t) > eps:
                    mi_t = Moran(t, w).I
                    moran_true_list.append(mi_t)
            mean_moran_pred = np.nanmean(moran_pred_list) if moran_pred_list else np.nan
            mean_moran_true = np.nanmean(moran_true_list) if moran_true_list else np.nan
            if not np.isnan(mean_moran_pred) and not np.isnan(mean_moran_true):
                mean_moran_diff = moran_score(mean_moran_pred , mean_moran_true)
        except Exception as e:
            print(f"[Warning] Moran's I failed: {e}")

    # ---- Assemble results ----
    results = {
        'metric': [
            'mean_spot_pcc', 'mean_spot_spearman',
            'mean_type_pcc', 'mean_type_spearman',
            'overall_mse', 'overall_rmse', 'mean_jsd',
            'mean_ssim', 'mean_moran_pred', 'mean_moran_true', 'mean_moran_diff'
        ],
        'value': [
            mean_spot_pcc, mean_spot_spearman,
            mean_type_pcc, mean_type_spearman,
            mse, rmse, mean_jsd,
            mean_ssim, mean_moran_pred, mean_moran_true, mean_moran_diff
        ]
    }
    df = pd.DataFrame(results).set_index('metric')
    return df


def _rasterize(coords, values, grid_size=64):
    """
    Rasterize scattered spatial values onto a 2D grid using nearest-neighbor assignment.
    Empty grid cells are filled with 0.
    """
    coords = np.asarray(coords)
    values = np.asarray(values)
    x, y = coords[:, 0], coords[:, 1]
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-12)
    xi = np.clip((x_norm * (grid_size - 1)).astype(int), 0, grid_size - 1)
    yi = np.clip((y_norm * (grid_size - 1)).astype(int), 0, grid_size - 1)

    grid = np.zeros((grid_size, grid_size), dtype=float)
    count = np.zeros((grid_size, grid_size), dtype=float)
    np.add.at(grid, (yi, xi), values)
    np.add.at(count, (yi, xi), 1)
    # Average per grid cell
    grid = np.divide(grid, count, out=np.zeros_like(grid), where=count > 0)
    return grid

/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#ours
import scanpy as sc
import pandas as pd
import numpy as np
import pandas as pd
cell_spot_gt = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_stereo-seq/spots_cells.csv',sep=',',header = 0,index_col=0)
import pandas as pd


/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [3]:
cell_spot_gt['spot_id']='X'+cell_spot_gt['spot_id'].values.astype('str')
spot_clean=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/spatial_coords.tsv",sep='\t',header=0,index_col=0)
#spot_clean.index='X'+spot_clean.index.astype(str)
matrix = pd.get_dummies(cell_spot_gt['spot_id']).astype(int)
A_gt=matrix.copy()
A_gt=A_gt[spot_clean.index]
count_matrix = cell_spot_gt.groupby(['spot_id', 'anno']).size().unstack(fill_value=0)
count_matrix=count_matrix.div(count_matrix.sum(axis=1), axis=0)
count_matrix=count_matrix.loc[spot_clean.index]
A_gt_deconv=count_matrix.copy()

A_pred=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/merfish_A_optimized_0425.csv',sep=',',index_col=0)
A_pred.index='X'+A_pred.index.astype(str)


In [4]:
A_optimized1=A_pred.copy()
A_optimized1[A_optimized1<0.1]=0
A_optimized1 = A_optimized1.div(A_optimized1.sum(axis=1), axis=0)
A_gt_deconv=A_gt_deconv.reindex(columns=A_pred.columns, fill_value=0)
df_results_ours=evaluate_deconvolution(A_optimized1.values, A_gt_deconv.values,coords=spot_clean)
celltype = pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/cell_type_labels.tsv",sep='\t',header = 0)


In [5]:
df_results_ours

,value
metric,
mean_spot_pcc,0.820678
mean_spot_spearman,0.650976
mean_type_pcc,0.874404
mean_type_spearman,0.719523
overall_mse,0.013015
overall_rmse,0.114082
mean_jsd,0.222039
mean_ssim,0.828139
mean_moran_pred,0.443592


In [6]:
# novosparc
celltype = pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/cell_type_labels.tsv",sep='\t',header = 0)
subclass=celltype['CellType'].values
df_onehot = pd.get_dummies(subclass) 
class_names = df_onehot.columns.tolist()
A_gt_deconv = A_gt_deconv.reindex(columns=class_names, fill_value=0)
novo = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/novosparc_normalization.npy') 

X = df_onehot.values.T 
print(f"矩阵 X 的形状: {X.shape}") # 应输出 (23, 4198)
result = np.dot(X, novo)
result=result.T
row_sums = result.sum(axis=1)  # 每个 spot 的总和，形状 (424,)
spot_ct_proportions = result / row_sums[:, np.newaxis]  # 或者 result / row_sums.reshape(-1,1)
df_results_novosparc = evaluate_deconvolution(spot_ct_proportions, A_gt_deconv.values,coords=spot_clean)


矩阵 X 的形状: (13, 2489)


In [34]:
df_results_novosparc


,value
metric,
mean_spot_pcc,0.223068
mean_spot_spearman,0.180352
mean_type_pcc,0.076621
mean_type_spearman,0.06
overall_mse,0.042177
overall_rmse,0.20537
mean_jsd,0.606939
mean_ssim,0.623059
mean_moran_pred,0.633802


In [7]:
# tangram
spatial_with_tangram=sc.read("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/spatial_with_tangram-Simulation.h5ad")
tangram=spatial_with_tangram.obsm["tangram_ct_pred"]
tangram=tangram[class_names].values
tangram=tangram/tangram.sum(axis=1,keepdims=True)
df_results_tangram = evaluate_deconvolution(tangram, A_gt_deconv.values,coords=spot_clean)


In [36]:
df_results_tangram

,value
metric,
mean_spot_pcc,0.6939
mean_spot_spearman,0.458959
mean_type_pcc,0.662923
mean_type_spearman,0.471718
overall_mse,0.024394
overall_rmse,0.156187
mean_jsd,0.386756
mean_ssim,0.728133
mean_moran_pred,0.268612


In [9]:
#spotiphy
A_dev_pred=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/Spotiphy_Result-SimulationData-zz/proportion2-simulationdata.csv",header=0,index_col=0)
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
df_results_spotiphy = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean)

In [38]:
df_results_spotiphy

,value
metric,
mean_spot_pcc,0.18638
mean_spot_spearman,0.138129
mean_type_pcc,0.21467
mean_type_spearman,0.13612
overall_mse,0.076626
overall_rmse,0.276813
mean_jsd,0.696981
mean_ssim,0.702104
mean_moran_pred,0.0678


In [39]:
# RCTD
pred_A=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/RCTD_results/RCTD_weights.csv",sep=',',header=0,index_col=0)
# 将列名中的 '/' 替换为 ' ' (空格)
gtcol =A_gt_deconv.columns
pred_A=pred_A[gtcol]
pred_A = pred_A.div(pred_A.sum(axis=1), axis=0)
df_results_RCTD = evaluate_deconvolution(pred_A, A_gt_deconv.loc[pred_A.index].values,coords=spot_clean.loc[pred_A.index])

In [40]:
df_results_RCTD

,value
metric,
mean_spot_pcc,0.515096
mean_spot_spearman,0.26292
mean_type_pcc,0.476678
mean_type_spearman,0.270739
overall_mse,0.034892
overall_rmse,0.186795
mean_jsd,0.425702
mean_ssim,0.756219
mean_moran_pred,0.208248


In [41]:
# CIBRT
pred_A=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/CIBERSORTx_stereo.txt",sep='\t',header=0,index_col=0)
# 将列名中的 '/' 替换为 ' ' (空格)
gtcol =A_gt_deconv.columns.str.replace(' ','_')
pred_A=pred_A[gtcol]
pred_A = pred_A.div(pred_A.sum(axis=1), axis=0)
df_results_CIBERSORTx = evaluate_deconvolution(pred_A, A_gt_deconv.loc[pred_A.index].values,coords=spot_clean.loc[pred_A.index])

In [42]:
# spatialDLWS
A_dev_pred=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/spatialDWLS/spatialDWLS_dec_zztry.csv",sep=',',index_col=0)
A_dev_pred.columns = A_dev_pred.columns.str.replace('_', ' ', regex=False)
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
A_dev_pred = A_dev_pred.fillna(0)
df_results_spatialDLWS = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean)

In [43]:
df_results_spatialDLWS

,value
metric,
mean_spot_pcc,0.135826
mean_spot_spearman,0.204124
mean_type_pcc,0.206522
mean_type_spearman,0.191981
overall_mse,0.060696
overall_rmse,0.246365
mean_jsd,0.675003
mean_ssim,0.615967
mean_moran_pred,0.077749


In [44]:
#cytospace
A_dev_pred=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/cytospace_results/fractional_abundances_by_spot.csv",sep=',',header=0,index_col=0)
#A_dev_pred.index=[str0[1:] for str0 in A_dev_pred.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
A_dev_pred = A_dev_pred.fillna(0)
df_results_cytospace = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.loc[A_dev_pred.index].values,coords=spot_clean.loc[A_dev_pred.index])

In [45]:
#SIMO
celltype.index=celltype['Cell IDs'].values
import pandas as pd
df = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/SMIO/Location_map1_zztry.csv', index_col=0)
matrix = df.pivot(index='cell', columns='spot', values='value')
matrix = matrix.fillna(0)

celltype=celltype.loc[matrix.index]
ct_onehot_df = pd.get_dummies(celltype['CellType']) 
spot_ct_proportions = matrix.T @ ct_onehot_df  # n_spots × n_types
row_sums = spot_ct_proportions.sum(axis=1)  # Series
spot_ct_proportions = spot_ct_proportions.div(row_sums, axis=0)  # 按
spot_ct_proportions=spot_ct_proportions[A_gt_deconv.columns]
spot_ct_proportions=spot_ct_proportions.loc[A_gt_deconv.index]
pred = spot_ct_proportions.values.astype(np.float64)
df_results_SIMO = evaluate_deconvolution(pred, A_gt_deconv.values,coords=spot_clean)

In [46]:
A_dev_pred=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/cell2location_results/q05_cell_abundance_w_sf.csv',index_col=0)
A_dev_pred.columns = A_dev_pred.columns.str.replace('q05cell_abundance_w_sf_', '')
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
A_dev_pred = A_dev_pred.div(A_dev_pred.sum(axis=1), axis=0)

In [47]:
df_results_cell2loc = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean)

In [75]:
df_all = pd.concat(
    [df_results_ours,df_results_novosparc, df_results_tangram,
     df_results_spotiphy, df_results_RCTD,df_results_cytospace, df_results_spatialDLWS,df_results_cell2loc,df_results_CIBERSORTx],
    axis=1,
    keys=['scMagNet', 'novoSpaRc', 'Tangram', 'Spotiphy','RCTD', 'CytoSPACE', 'SpatialDWLS','cell2location','CIBERSORTx']
)

In [76]:
df_all.T.to_csv('deconvresult.txt',sep='\t')

## cell to spot mapping 

In [77]:
import numpy as np

def get_entropy_uniformity(matrix):
    # 防止 log(0) 报错，加上极小值
    epsilon = 1e-12
    matrix = np.clip(matrix, epsilon, 1)
    
    N = matrix.shape[0] # 行数
    # 计算每列的熵
    entropy = -np.sum(matrix * np.log(matrix), axis=0)
    # 归一化 (除以最大熵 log(N))
    max_entropy = np.log(N)
    uniformity_score = entropy / max_entropy
    
    return uniformity_score


In [78]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
def evaluate_mapping(pred_probs, gt_matrix, spot_coords, k_list=[1,3,5,10]):
    """
    评估 cell-to-spot mapping 的指标。
    
    参数:
        pred_probs: (n_cells, n_spots) 预测概率矩阵
        gt_matrix: (n_cells, n_spots) ground truth 二值矩阵，每行一个1
        spot_coords: (n_spots, 2) spot 坐标
        k_list: list of int, Top-k 准确率的 k 值
    
    返回:
        metrics: dict
    """
    # 确保是 numpy 数组
    pred_probs = np.asarray(pred_probs)
    gt_matrix = np.asarray(gt_matrix)
    spot_coords = np.asarray(spot_coords)
    
    n_cells, n_spots = pred_probs.shape
    assert gt_matrix.shape == (n_cells, n_spots), "形状不匹配"
    assert spot_coords.shape[0] == n_spots, "spot 坐标数量不匹配"
    
    # 真实标签：每行的 argmax，如果全零则忽略
    true_labels = np.argmax(gt_matrix, axis=1)
    valid_mask = gt_matrix.sum(axis=1) > 0
    if not np.all(valid_mask):
        print(f"警告：{np.sum(~valid_mask)} 个细胞在 ground truth 中无标签，已忽略。")
    true_labels = true_labels[valid_mask]
    pred_probs_valid = pred_probs[valid_mask]
    
    n_valid = len(true_labels)
    if n_valid == 0:
        return {}
    
    # 预测标签
    pred_labels = np.argmax(pred_probs_valid, axis=1)
    
    # 1. Mapping accuracy
    mapping_acc = np.mean(pred_labels == true_labels)
    
    # 2. Top-k accuracy
    topk_acc = {}
    for k in k_list:
        if k > n_spots:
            k = n_spots
        # 获取每个细胞预测概率最高的 k 个 spot 索引
        topk_indices = np.argsort(-pred_probs_valid, axis=1)[:, :k]
        # 检查真实标签是否在其中
        correct = np.any(topk_indices == true_labels[:, None], axis=1)
        topk_acc[f'top{k}_acc'] = np.mean(correct)
    
    # 3. AUROC / AUPRC (宏平均)
    
    y_true_flat = gt_matrix.flatten()
    y_score_flat = pred_probs.flatten()# Filter out pairs where gt is 0 AND keep all
    macro_auroc = roc_auc_score(y_true_flat, y_score_flat)
    macro_auprc = average_precision_score(y_true_flat, y_score_flat)
    # 4. 坐标 MAE / RMSE
    # 真实坐标
    true_coords = spot_coords[true_labels]
    # 预测坐标（软分配：加权平均）
    # 归一化预测概率（确保每行和为1）
    pred_probs_norm = pred_probs_valid / (pred_probs_valid.sum(axis=1, keepdims=True) + 1e-12)
    pred_coords = pred_probs_norm @ spot_coords  # (n_valid, 2)
    # 欧氏距离
    dists = np.linalg.norm(true_coords - pred_coords, axis=1)
    coord_mae = np.mean(dists)
    coord_rmse = np.sqrt(np.mean(dists**2))
    
    # 汇总
    metrics = {
        'mapping_accuracy': mapping_acc,
        'coord_mae': coord_mae,
        'coord_rmse': coord_rmse,
        'macro_auroc': macro_auroc,
        'macro_auprc': macro_auprc,
    }
    metrics.update(topk_acc)
    return metrics
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    jaccard_score
)

In [52]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    jaccard_score
)
def evaluate_mapping_pairs(gt_matrix, pred_pairs,
                           cell_col="cellid", spot_col="spotname",
                           cell_order=None, spot_order=None):
    """
    Evaluate cell-to-spot mapping predictions against a binary ground truth matrix.

    Parameters
    ----------
    gt_matrix : np.ndarray or pd.DataFrame, shape (n_cells, n_spots)
        Binary ground-truth matrix. Each row should have exactly one 1.
    pred_pairs : pd.DataFrame
        Two columns: cell ID and spot name for predicted pairs.
    cell_col, spot_col : str
        Column names in pred_pairs indicating cell and spot IDs.
    cell_order, spot_order : list, optional
        Explicit row/column order for gt_matrix. If None, taken from gt_matrix.

    Returns
    -------
    pd.DataFrame with metrics.
    """
    # ---------- 1. Standardize gt_matrix ----------
    if isinstance(gt_matrix, pd.DataFrame):
        gt_df = gt_matrix.copy()
    else:
        gt_df = pd.DataFrame(gt_matrix)

    if cell_order is None:
        cell_order = list(gt_df.index)
    if spot_order is None:
        spot_order = list(gt_df.columns)

    gt_df = gt_df.reindex(index=cell_order, columns=spot_order).fillna(0)
    gt_arr = (gt_df.values > 0).astype(int)

    # ---------- 2. Build predicted binary matrix ----------
    pred_mat = np.zeros_like(gt_arr)

    # Map cell/spot IDs to row/col index
    cell_to_idx = {c: i for i, c in enumerate(cell_order)}
    spot_to_idx = {s: j for j, s in enumerate(spot_order)}

    # Counters for pairs outside gt space
    n_missing_cell = 0
    n_missing_spot = 0
    n_valid_pairs = 0

    for _, row in pred_pairs.iterrows():
        c = row[cell_col]
        s = row[spot_col]
        if c not in cell_to_idx:
            n_missing_cell += 1
            continue
        if s not in spot_to_idx:
            n_missing_spot += 1
            continue
        pred_mat[cell_to_idx[c], spot_to_idx[s]] = 1
        n_valid_pairs += 1

    # ---------- 3. Flatten to 1D labels ----------
    y_true = gt_arr.flatten()
    y_pred = pred_mat.flatten()

    # ---------- 4. Metrics ----------
    acc = accuracy_score(y_true, y_pred)

    # For binary one-vs-rest style metrics, use pos_label=1
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    # Jaccard on binary labels (equivalent to TP / (TP + FP + FN))
    jacc = jaccard_score(y_true, y_pred, pos_label=1, zero_division=0)

    # Additional counts for reference
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    results = {
        "metric": [
            "accuracy", "precision", "recall", "f1", "jaccard",
            "TP", "FP", "FN", "TN",
            "n_gt_positives", "n_pred_positives",
            "n_valid_pairs", "n_missing_cell", "n_missing_spot"
        ],
        "value": [
            acc, prec, rec, f1, jacc,
            tp, fp, fn, tn,
            int(y_true.sum()), int(y_pred.sum()),
            n_valid_pairs, n_missing_cell, n_missing_spot
        ]
    }
    return pd.DataFrame(results).set_index("metric")

In [79]:
pred_A=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/Ours/predicted_A_matrix.csv",sep=',',header=0,index_col=0)

In [80]:
spatial_coords=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/spatial_coords.tsv', sep='\t')
spatial_coords.index=spatial_coords['SpotID'].values
#spatial_coords=spatial_coords.loc[pred_A.index]
adata_sc=sc.read('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_stereo-seq/adata_sc_Xenium.h5ad')
matrix = pd.get_dummies(cell_spot_gt['spot_id']).astype(int)
A_gt=matrix.copy()
A_gt=A_gt[spot_clean.index]
A_gt_reindexed = A_gt.reindex(adata_sc.obs_names, fill_value=0)
A_gt=A_gt_reindexed.copy()

In [81]:
A_gt=A_gt.T

In [82]:
assigned_cells = pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_stereo-seq/spots_cells.csv", index_col=0,header=0)
assigned_cells['cell_id']=assigned_cells.index
df_result = pd.crosstab(assigned_cells['spot_id'], assigned_cells['cell_id'])
spots_meta_df=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_stereo-seq/spots_info.csv", header=0)
df_result1 = pd.DataFrame(np.zeros((spots_meta_df.shape[0],A_gt.shape[1])), index=spots_meta_df['spot_id'], columns=pred_A.columns)
df_result1[df_result.columns]= df_result.values
df_result1.index=df_result.index

metrics_ours=evaluate_mapping(pred_A.T,df_result1.T,spatial_coords[['row','col']].values)


警告：603 个细胞在 ground truth 中无标签，已忽略。


In [57]:
metrics_ours

{'mapping_accuracy': np.float64(0.690880169671262),
 'coord_mae': np.float64(153.88542668255604),
 'coord_rmse': np.float64(187.84256941887685),
 'macro_auroc': 0.9857835650366125,
 'macro_auprc': 0.6550283485967171,
 'top1_acc': np.float64(0.690880169671262),
 'top3_acc': np.float64(0.8303287380699894),
 'top5_acc': np.float64(0.8753976670201484),
 'top10_acc': np.float64(0.9215270413573701)}

In [58]:
#novosparc
novo = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/novosparc_normalization.npy') 

metrics_novo=evaluate_mapping(novo,df_result1.T,spatial_coords[['row','col']].values)

警告：603 个细胞在 ground truth 中无标签，已忽略。


In [59]:
metrics_novo

{'mapping_accuracy': np.float64(0.04612937433722163),
 'coord_mae': np.float64(264.23803348759253),
 'coord_rmse': np.float64(283.7279320462318),
 'macro_auroc': 0.6353606624260161,
 'macro_auprc': 0.006428907878266672,
 'top1_acc': np.float64(0.04612937433722163),
 'top3_acc': np.float64(0.09862142099681867),
 'top5_acc': np.float64(0.1383881230116649),
 'top10_acc': np.float64(0.21367974549310711)}

In [60]:
#tangram
import scanpy as sc
mapping_tangram=sc.read('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/tangram_mapping-Simulation.h5ad')
metrics_tangram=evaluate_mapping(mapping_tangram.X,df_result1.T,spatial_coords[['row','col']].values)
metrics_tangram

警告：603 个细胞在 ground truth 中无标签，已忽略。


{'mapping_accuracy': np.float64(0.3361611876988335),
 'coord_mae': np.float64(213.51737745139124),
 'coord_rmse': np.float64(253.2748563713516),
 'macro_auroc': 0.897658420962818,
 'macro_auprc': 0.21569130511557008,
 'top1_acc': np.float64(0.3361611876988335),
 'top3_acc': np.float64(0.4851537645811241),
 'top5_acc': np.float64(0.5583244962884412),
 'top10_acc': np.float64(0.6648992576882291)}

In [61]:
#SIMO
import pandas as pd
df = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/SMIO/Location_map1_zztry.csv', index_col=0)
df_result1.index='X'+df_result1.index.astype(str)
metrics1_SIMO=evaluate_mapping_pairs(df_result1.T,df,spot_col='spot',cell_col='cell')

In [62]:
import pandas as pd
import numpy as np

N = df.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_SIMO=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')

In [63]:
metrics1_Ours_SIMO

,value
metric,
accuracy,0.997691
precision,0.794470
recall,0.457052
f1,0.580276
jaccard,0.408725
TP,862.000000
FP,223.000000
FN,1024.000000
TN,538004.000000


In [64]:
#cytospace
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/cytospace_results/assigned_locations.csv')
metrics1_cytoscape=evaluate_mapping_pairs(df_result1.T,df,spot_col='SpotID',cell_col='OriginalCID')


In [65]:
metrics1_cytoscape

,value
metric,
accuracy,0.996103
precision,0.388379
recall,0.202015
f1,0.265783
jaccard,0.153258
TP,381.000000
FP,600.000000
FN,1505.000000
TN,537627.000000


In [66]:
import pandas as pd
import numpy as np

N = df.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_cytospace=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [67]:
# CellTrek
matrix=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/CellTreck_assignment.txt',sep=' ',header=0,index_col=0)
import pandas as pd
import numpy as np
# Option 1: Using stack (works for both sparse and dense)
stacked = matrix.stack()
pairs = stacked[stacked >0].reset_index()
pairs.columns = ['cellid', 'spotname','value']
new_index = [ str0.split('_')[-1]+'-'+str0.split('_')[0]+'-'+'_'.join(str0.split('_')[1:-1]) for str0 in pairs['cellid'].values]
pairs['cellid']=new_index
metrics1_CellTrek=evaluate_mapping_pairs(df_result1.T,pairs,spot_col='spotname',cell_col='cellid')

metrics1_CellTrek

,value
metric,
accuracy,0.994023
precision,0.041667
recall,0.032344
f1,0.036418
jaccard,0.018547
TP,61.000000
FP,1403.000000
FN,1825.000000
TN,536824.000000


In [68]:
pairs.shape

(1464, 3)

In [69]:
import pandas as pd
import numpy as np

N = pairs.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_CellTrek=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [70]:
#CMAp
# CellTrek
matrix=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/CMAP/cell_to_spot.txt',sep=' ',header=0,index_col=0)
# Option 1: Using stack (works for both sparse and dense)
stacked = matrix.stack()
pairs = stacked[stacked >0].reset_index()
pairs.columns = ['cellid', 'spotname','value']
metrics1_CMAP=evaluate_mapping_pairs(df_result1.T,pairs,spot_col='spotname',cell_col='cellid')


In [71]:
pairs.shape

(1748, 3)

In [72]:
import pandas as pd
import numpy as np

N = pairs.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_CMAP=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [73]:
metrics1_CMAP=evaluate_mapping_pairs(df_result1.T,pairs,spot_col='spotname',cell_col='cellid')

In [74]:
metrics_CMAP

NameError: name 'metrics_CMAP' is not defined

In [ ]:
# spaTrio
df = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/spaTrio/output.csv', sep=',',index_col=0)
metrics1_SpaTrio=evaluate_mapping_pairs(df_result1.T,df,spot_col='spot',cell_col='cell')

In [ ]:
import pandas as pd
import numpy as np

N = df.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_spaTrio=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [ ]:
metrics1_Ours_spaTrio

,value
metric,
accuracy,0.997691
precision,0.794470
recall,0.457052
f1,0.580276
jaccard,0.408725
TP,862.000000
FP,223.000000
FN,1024.000000
TN,538004.000000


In [ ]:
import pandas as pd
import numpy as np

# 1. 将所有指标的字典放入一个列表中
metrics_list = [
    metrics_ours, 
    metrics_tangram, 
    metrics_novo, 
]

# 2. 定义与列表顺序对应的标签名称
method_names = [
    "ours", 
    "Tangram", 
    "NovoSpa", 
]
# 3. 使用列表推导式，将字典批量转换为带有 index=[0] 的单行 DataFrame
# 这样每个 DataFrame 都是 1行 x N列 的结构，可以直接横向拼接
df_list = [pd.DataFrame(m, index=[0]) for m in metrics_list]

# 4. 横向合并 (axis=1)，并使用 keys 参数添加方法标签作为多级索引
metrics_all = pd.concat(df_list, axis=0, keys=method_names)

# 5. (可选) 扁平化列名，方便后续导出为 Excel 或画图


In [ ]:
metrics_all.index=  [f"{method}" for method, metric in metrics_all.T.columns]


In [ ]:
metrics_all.T.to_csv('cell_to_spot_result.csv')

In [ ]:
metrics_all.T

,ours,Tangram,NovoSpa
mapping_accuracy,0.690880,0.336161,0.046129
coord_mae,153.885427,213.517377,264.238033
coord_rmse,187.842569,253.274856,283.727932
macro_auroc,0.985784,0.897658,0.635361
macro_auprc,0.655028,0.215691,0.006429
top1_acc,0.690880,0.336161,0.046129
top3_acc,0.830329,0.485154,0.098621
top5_acc,0.875398,0.558324,0.138388
top10_acc,0.921527,0.664899,0.213680


In [ ]:
import pandas as pd
import numpy as np

# 1. 将所有指标的字典放入一个列表中
metrics_list = [
    metrics1_cytoscape, 
    metrics1_Ours_cytospace, 
    metrics1_CellTrek,
    metrics1_Ours_CellTrek, 
    metrics1_CMAP, 
    metrics1_Ours_CMAP, 
    metrics1_SIMO, 
    metrics1_Ours_SIMO, 
    metrics1_SpaTrio, 
    metrics1_Ours_spaTrio
]

# 2. 定义与列表顺序对应的标签名称
method_names = [
    "CytoSPACE", 
    'Ours0',
    "CellTrek", 
    'Ours1',
    "CMAP", 
    'Ours2',
    "SIMO", 
    'Ours3',
    "SpaTrio",
    'Ours4',
]
# 3. 使用列表推导式，将字典批量转换为带有 index=[0] 的单行 DataFrame
# 这样每个 DataFrame 都是 1行 x N列 的结构，可以直接横向拼接

# 4. 横向合并 (axis=1)，并使用 keys 参数添加方法标签作为多级索引
metrics_all = pd.concat(metrics_list, axis=1, keys=method_names)

# 5. (可选) 扁平化列名，方便后续导出为 Excel 或画图


(1748, 217)

In [ ]:
metrics_all.to_csv('cell_asignment.txt',sep='\t')


{'mapping_accuracy': np.float64(0.3361611876988335),
 'coord_mae': np.float64(213.51737745139124),
 'coord_rmse': np.float64(253.2748563713516),
 'macro_auroc': 0.897658420962818,
 'macro_auprc': 0.21569130511557008,
 'top1_acc': np.float64(0.3361611876988335),
 'top3_acc': np.float64(0.4851537645811241),
 'top5_acc': np.float64(0.5583244962884412),
 'top10_acc': np.float64(0.6648992576882291)}

In [83]:
## stalocator
import numpy as np
cell_loc=np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/STALocator/map_A_sc.npy')
spot_loc = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/STALocator/map_B_st.npy')

In [84]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
import pandas as pd

# 假设 spot_loc 和 cell_loc 是 numpy array，形状分别为 (477, 2) 和 (16068, 2)
spot_loc = np.asarray(spot_loc)
cell_loc = np.asarray(cell_loc)

n_cells = cell_loc.shape[0]
n_spots = spot_loc.shape[0]
k = 10  # 每个 cell 找最近的 10 个 spot

# 1. 构建 KDTree
tree = cKDTree(spot_loc)

# 2. 查询每个 cell 最近的 k 个 spot
# distances: (n_cells, k), indices: (n_cells, k)
distances, indices = tree.query(cell_loc, k=k)

# 3. 权重：top1=10, top2=9, ..., top10=1
weights = np.arange(k, 0, -1)  # [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]

# 4. 构建稀疏矩阵 (n_cells × n_spots)
rows = np.repeat(np.arange(n_cells), k)
cols = indices.ravel()
vals = np.tile(weights, n_cells)

M = csr_matrix((vals, (rows, cols)), shape=(n_cells, n_spots))

# 如果需要稠密 DataFrame：
# df = pd.DataFrame(M.toarray())

print(M.shape)   # (16068, 477)
print(M.nnz)     # 16068 * 10 = 160680 个非零值

(2489, 217)
24890


In [86]:
metrics_STALocator=evaluate_mapping(M.toarray(),df_result1.T,spatial_coords[['row','col']].values)

警告：603 个细胞在 ground truth 中无标签，已忽略。


In [ ]:
metrics_STALocator

{'mapping_accuracy': np.float64(0.005302226935312832),
 'coord_mae': np.float64(285.77034583875485),
 'coord_rmse': np.float64(340.739715319935),
 'macro_auroc': 0.5090380150225814,
 'macro_auprc': 0.0035674093998437394,
 'top1_acc': np.float64(0.005302226935312832),
 'top3_acc': np.float64(0.016967126193001062),
 'top5_acc': np.float64(0.030752916224814422),
 'top10_acc': np.float64(0.06415694591728525)}

: 